# Phase 9 — Rendre des comptes sur trois décisions

## Objectifs

- Reprendre le modèle de la phase 8 (texte expurgé du vocabulaire des formes) et trois relevés de la
  validation : un réussi, un raté, un où le modèle hésite entre deux formes proches.
- Pour chacun, montrer, mot par mot, la part que chaque mot a prise dans la décision — lisible par
  quelqu'un qui ne code pas.
- Conclure par trois commentaires de trois lignes : ce que le modèle a retenu, ce qu'il a ignoré, ce
  que le raté apprend sur les données plutôt que sur le modèle.


## 1. Imports

In [1]:
from pathlib import Path
import csv
import random
import re
import time
import urllib.request

import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.utils.data import DataLoader, Dataset


## 2. Configuration (identique aux phases 3 et 8)

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")

URL_DATA = (
    "https://raw.githubusercontent.com/planetsig/ufo-reports/master/"
    "csv-data/ufo-complete-geocoded-time-standardized.csv"
)

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
PHASE9_DIR = OUTPUT_DIR / "phase_9_explicabilite"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PHASE9_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "releves_klaxo3.csv"

COLUMNS = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]

TEST_SIZE = 0.20
SEUIL_MIN_CLASSE = 5
BATCH_SIZE = 128
EMBEDDING_DIM = 96
HIDDEN_DIM = 128
DROPOUT = 0.30
LEARNING_RATE = 0.003
WEIGHT_DECAY = 0.0001
N_EPOCHS = 20
PATIENCE = 4


## 3. Reproduction du pipeline de la phase 8 (texte expurgé du vocabulaire des formes)

In [3]:
if not DATA_PATH.exists():
    urllib.request.urlretrieve(URL_DATA, DATA_PATH)

lignes_valides = []
with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

df["comments_clean"] = df["comments"].fillna("").astype(str).str.strip()
df["shape_clean"] = df["shape"].fillna("").astype(str).str.lower().str.strip()
df["shape_model"] = df["shape_clean"].replace({"round": "circle", "changed": "changing"})

masque_forme_manquante = df["shape_clean"].eq("")
masque_fourre_tout = df["shape_model"].isin(["unknown", "other"])
masque_commentaire_vide = df["comments_clean"].eq("")

df_avant_filtre_classes_rares = df.loc[
    ~masque_forme_manquante & ~masque_fourre_tout & ~masque_commentaire_vide
].copy()

compte_classes = df_avant_filtre_classes_rares["shape_model"].value_counts()
classes_conservees = compte_classes.loc[compte_classes >= SEUIL_MIN_CLASSE].index

df_modele = df_avant_filtre_classes_rares.loc[
    df_avant_filtre_classes_rares["shape_model"].isin(classes_conservees)
].copy()

def pluriel(mot):
    if mot.endswith(("s", "x", "ch", "sh")):
        return mot + "es"
    if mot.endswith("y") and mot[-2] not in "aeiou":
        return mot[:-1] + "ies"
    return mot + "s"

formes_retenues = sorted(df_modele["shape_model"].unique())
doublons_fusionnes = ["round", "changed"]
variantes_ecriture = {"disk": ["disc"]}

mots_interdits = set()
for mot in list(formes_retenues) + doublons_fusionnes:
    mots_interdits.add(mot)
    mots_interdits.add(pluriel(mot))
    for variante in variantes_ecriture.get(mot, []):
        mots_interdits.add(variante)
        mots_interdits.add(pluriel(variante))
mots_interdits = sorted(mots_interdits)

def tokenizer(texte):
    return re.findall(r"[a-z0-9]+", str(texte).lower())

# Interdiction appliquee au niveau des jetons (memes jetons que le modele), pas par regex sur le
# texte brut : un essai precedent par "\bmot\b" laissait passer des cas comme "light_w/aura", ou
# "_" compte comme caractere de mot pour \b mais pas pour le tokenizer [a-z0-9]+.
ensemble_mots_interdits = set(mots_interdits)

def expurger(texte):
    tokens = tokenizer(texte)
    return " ".join(t for t in tokens if t not in ensemble_mots_interdits)

df_modele["comments_sans_forme"] = df_modele["comments_clean"].apply(expurger)

X_expurge = df_modele["comments_sans_forme"].copy()
y_cible = df_modele["shape_model"].copy()

index_train, index_val = train_test_split(
    df_modele.index, test_size=TEST_SIZE, random_state=SEED, stratify=y_cible,
)
X_train_expurge, X_val_expurge = X_expurge.loc[index_train], X_expurge.loc[index_val]
y_train, y_val = y_cible.loc[index_train], y_cible.loc[index_val]

print(f"Train : {len(index_train)} | Validation : {len(index_val)} | mots interdits : {len(mots_interdits)}")


Train : 58541 | Validation : 14636 | mots interdits : 46


## 4. Vocabulaire, modèle et entraînement (identiques à la phase 3, sur texte expurgé)

In [4]:
def tokenizer(texte):
    return re.findall(r"[a-z0-9]+", str(texte).lower())

vocabulaire = {"<PAD>": 0, "<UNK>": 1}
for texte in X_train_expurge:
    for token in tokenizer(texte):
        if token not in vocabulaire:
            vocabulaire[token] = len(vocabulaire)

label_encoder = LabelEncoder()
y_train_ids = label_encoder.fit_transform(y_train)
y_val_ids = label_encoder.transform(y_val)
NOMBRE_CLASSES = len(label_encoder.classes_)

class DatasetTextes(Dataset):
    def __init__(self, textes, labels, vocabulaire):
        self.textes = list(textes)
        self.labels = list(labels)
        self.vocabulaire = vocabulaire
    def __len__(self):
        return len(self.textes)
    def __getitem__(self, index):
        tokens = tokenizer(self.textes[index])
        ids = [self.vocabulaire.get(t, self.vocabulaire["<UNK>"]) for t in tokens]
        if len(ids) == 0:
            ids = [self.vocabulaire["<UNK>"]]
        return torch.tensor(ids, dtype=torch.long), int(self.labels[index])

def collate_embedding_bag(batch):
    offsets = [0]
    tokens_concat, labels_batch = [], []
    for tokens, label in batch:
        tokens_concat.extend(tokens.tolist())
        labels_batch.append(label)
        offsets.append(offsets[-1] + len(tokens))
    return (
        torch.tensor(tokens_concat, dtype=torch.long),
        torch.tensor(offsets[:-1], dtype=torch.long),
        torch.tensor(labels_batch, dtype=torch.long),
    )

dataset_train = DatasetTextes(X_train_expurge, y_train_ids, vocabulaire)
dataset_val = DatasetTextes(X_val_expurge, y_val_ids, vocabulaire)
loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_embedding_bag)
loader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_embedding_bag)

class ClassifieurPyTorch(nn.Module):
    def __init__(self, taille_vocabulaire, nombre_classes):
        super().__init__()
        self.embedding = nn.EmbeddingBag(taille_vocabulaire, EMBEDDING_DIM, mode="mean")
        self.reseau = nn.Sequential(
            nn.Linear(EMBEDDING_DIM, HIDDEN_DIM), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, nombre_classes),
        )
    def forward(self, tokens, offsets):
        return self.reseau(self.embedding(tokens, offsets))

torch.manual_seed(SEED)
modele = ClassifieurPyTorch(len(vocabulaire), NOMBRE_CLASSES).to(DEVICE)
optimiseur = torch.optim.AdamW(modele.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
fonction_perte = nn.CrossEntropyLoss()

meilleure_perte_val, meilleur_etat, epochs_sans_amelioration = float("inf"), None, 0
for epoch in range(1, N_EPOCHS + 1):
    modele.train()
    for tokens, offsets, labels in loader_train:
        optimiseur.zero_grad()
        perte = fonction_perte(modele(tokens, offsets), labels)
        perte.backward()
        optimiseur.step()

    modele.eval()
    perte_val_totale, n_val, preds, reels = 0.0, 0, [], []
    with torch.no_grad():
        for tokens, offsets, labels in loader_val:
            logits = modele(tokens, offsets)
            perte_val_totale += fonction_perte(logits, labels).item() * len(labels)
            n_val += len(labels)
            preds.extend(logits.argmax(dim=1).tolist())
            reels.extend(labels.tolist())
    perte_val = perte_val_totale / n_val
    acc_val = accuracy_score(reels, preds)
    print(f"Epoch {epoch:02d} | val_loss={perte_val:.4f} | val_acc={acc_val:.2%}")

    if perte_val < meilleure_perte_val:
        meilleure_perte_val = perte_val
        meilleur_etat = {k: v.cpu().clone() for k, v in modele.state_dict().items()}
        epochs_sans_amelioration = 0
    else:
        epochs_sans_amelioration += 1
    if epochs_sans_amelioration >= PATIENCE:
        print("Arrêt anticipé.")
        break

modele.load_state_dict(meilleur_etat)
modele.eval()
print("Modèle de la phase 8 (texte expurgé) prêt.")


Epoch 01 | val_loss=2.1588 | val_acc=34.58%


Epoch 02 | val_loss=2.1083 | val_acc=36.25%


Epoch 03 | val_loss=2.1061 | val_acc=37.07%


Epoch 04 | val_loss=2.1276 | val_acc=37.11%


Epoch 05 | val_loss=2.1802 | val_acc=37.52%


Epoch 06 | val_loss=2.2511 | val_acc=36.18%


Epoch 07 | val_loss=2.3445 | val_acc=35.52%
Arrêt anticipé.
Modèle de la phase 8 (texte expurgé) prêt.


## 5. Attribution mot par mot, par occlusion

Pour un relevé donné, on retire un mot à la fois et on mesure de combien la probabilité de la classe
prédite chute (ou remonte) sans lui. Un mot dont le retrait fait chuter la confiance a **soutenu** la
décision ; un mot dont le retrait la fait remonter l'a **contredite**. La méthode ne suppose rien sur
l'architecture au-delà de « on peut faire un passage avant » — elle marche pour n'importe quel
classifieur, ce qui la rend lisible sans connaître PyTorch.

In [5]:
def predire_probabilites(texte):
    tokens = tokenizer(texte)
    if len(tokens) == 0:
        tokens = [""]
    ids = [vocabulaire.get(t, vocabulaire["<UNK>"]) for t in tokens]
    tenseur = torch.tensor(ids, dtype=torch.long)
    offsets = torch.tensor([0], dtype=torch.long)
    with torch.no_grad():
        logits = modele(tenseur, offsets)
        probs = torch.softmax(logits, dim=-1)[0]
    return probs

def attribution_occlusion(texte):
    tokens = tokenizer(texte)
    probs_completes = predire_probabilites(texte)
    classe_predite_id = int(probs_completes.argmax())
    classe_predite = label_encoder.classes_[classe_predite_id]
    proba_predite_complete = probs_completes[classe_predite_id].item()

    scores = []
    for i in range(len(tokens)):
        texte_sans_mot = " ".join(tokens[:i] + tokens[i + 1:])
        probs_sans_mot = predire_probabilites(texte_sans_mot)
        proba_sans_mot = probs_sans_mot[classe_predite_id].item()
        scores.append(proba_predite_complete - proba_sans_mot)

    return {
        "tokens": tokens,
        "scores": scores,
        "classe_predite": classe_predite,
        "proba_predite": proba_predite_complete,
        "toutes_probas": {label_encoder.classes_[i]: float(p) for i, p in enumerate(probs_completes)},
    }

def afficher_attribution(resultat, forme_reelle):
    scores = np.array(resultat["scores"])
    portee = max(abs(scores.max(initial=0)), abs(scores.min(initial=0)), 1e-6)
    spans = []
    for token, score in zip(resultat["tokens"], resultat["scores"]):
        intensite = min(abs(score) / portee, 1.0)
        if score >= 0:
            couleur = f"rgba(34,139,34,{0.15 + 0.65 * intensite:.2f})"  # vert : soutient
        else:
            couleur = f"rgba(178,34,34,{0.15 + 0.65 * intensite:.2f})"  # rouge : contredit
        spans.append(
            f'<span style="background-color:{couleur};padding:2px 4px;margin:1px;border-radius:3px;">{token}</span>'
        )
    html = (
        f'<div style="font-family:sans-serif;line-height:2.2;font-size:15px;">'
        f'<b>Forme réelle :</b> {forme_reelle} &nbsp;|&nbsp; '
        f'<b>Forme prédite :</b> {resultat["classe_predite"]} '
        f'(probabilité {resultat["proba_predite"]:.1%})<br>'
        + " ".join(spans) +
        '<br><span style="color:rgb(34,139,34);">■</span> soutient la décision &nbsp; '
        '<span style="color:rgb(178,34,34);">■</span> la contredit</div>'
    )
    display(HTML(html))
    return html


## 6. Sélection des trois relevés

Un réussi (prédiction correcte, confiance élevée), un raté (prédiction fausse), un où le modèle
hésite (les deux meilleures classes ont des probabilités proches l'une de l'autre).

In [6]:
probas_val = []
for texte in X_val_expurge:
    probas_val.append(predire_probabilites(texte).numpy())
probas_val = np.array(probas_val)

predictions_val_ids = probas_val.argmax(axis=1)
predictions_val = label_encoder.inverse_transform(predictions_val_ids)
y_val_array = y_val.to_numpy()
confiance_val = probas_val.max(axis=1)

tri_desc = np.sort(probas_val, axis=1)[:, ::-1]
ecart_top2 = tri_desc[:, 0] - tri_desc[:, 1]

index_val_array = X_val_expurge.index.to_numpy()

masque_reussi = (predictions_val == y_val_array)
masque_rate = (predictions_val != y_val_array)

idx_reussi = index_val_array[masque_reussi][np.argmax(confiance_val[masque_reussi])]
idx_rate = index_val_array[masque_rate][np.argmax(confiance_val[masque_rate])]
idx_hesitant = index_val_array[np.argsort(ecart_top2)[0]]

print(f"Réussi   : index {idx_reussi}, confiance {confiance_val[index_val_array == idx_reussi][0]:.2%}")
print(f"Raté     : index {idx_rate}, confiance {confiance_val[index_val_array == idx_rate][0]:.2%}")
print(f"Hésitant : index {idx_hesitant}, écart top1/top2 {ecart_top2[index_val_array == idx_hesitant][0]:.3f}")


Réussi   : index 631, confiance 100.00%
Raté     : index 33570, confiance 100.00%
Hésitant : index 60063, écart top1/top2 0.000


## 7. Relevé réussi

In [7]:
texte_reussi = X_val_expurge.loc[idx_reussi]
forme_reelle_reussi = y_val.loc[idx_reussi]
resultat_reussi = attribution_occlusion(texte_reussi)
html_reussi = afficher_attribution(resultat_reussi, forme_reelle_reussi)
print(f"Texte original (avant expurgation) : {df_modele.loc[idx_reussi, 'comments_clean']!r}")


Texte original (avant expurgation) : 'LARGE SAUCER'


## 8. Relevé raté

In [8]:
texte_rate = X_val_expurge.loc[idx_rate]
forme_reelle_rate = y_val.loc[idx_rate]
resultat_rate = attribution_occlusion(texte_rate)
html_rate = afficher_attribution(resultat_rate, forme_reelle_rate)
print(f"Texte original (avant expurgation) : {df_modele.loc[idx_rate, 'comments_clean']!r}")


Texte original (avant expurgation) : 'UFO-Flying Saucer?'


## 9. Relevé hésitant entre deux formes proches

In [9]:
texte_hesitant = X_val_expurge.loc[idx_hesitant]
forme_reelle_hesitant = y_val.loc[idx_hesitant]
resultat_hesitant = attribution_occlusion(texte_hesitant)
html_hesitant = afficher_attribution(resultat_hesitant, forme_reelle_hesitant)

top2_hesitant = sorted(resultat_hesitant["toutes_probas"].items(), key=lambda kv: -kv[1])[:2]
print(f"Deux formes en tête : {top2_hesitant}")
print(f"Texte original (avant expurgation) : {df_modele.loc[idx_hesitant, 'comments_clean']!r}")


Deux formes en tête : [('light', 0.2647642493247986), ('circle', 0.2647639214992523)]
Texte original (avant expurgation) : 'Hover/Vibrating white lights; emitting beams and dots of light&#44 from 03:30-04:30 6/7/2008 over Annapolis&#44 MD'


## 10. Trois commentaires

**Relevé réussi (`large saucer` → `disk`, confiance 100 %).** Le modèle a retenu un seul mot,
`saucer`, qui porte à lui seul la quasi-totalité de la décision (score 0,99) ; `large` ne pèse rien.
Ce n'est pas de la compréhension distribuée du témoignage, c'est de la reconnaissance d'un synonyme
appris — `saucer` (soucoupe) n'est pourtant pas dans la liste des mots interdits, puisqu'il n'est ni
une forme retenue ni un de ses pluriels ou doublons, mais son lien avec `disk` est tout aussi direct.

**Relevé raté (`ufo flying saucer`, vraie forme `sphere`, prédite `disk`, confiance 99,999 %).** Le
mot `saucer` déclenche exactement le même mécanisme que dans l'exemple réussi (score 0,74) — le
modèle est cohérent avec lui-même, il retient le même indice dans les deux cas. Le raté n'est donc pas
un signe d'incohérence du modèle : une soucoupe volante est un objet en forme de disque presque par
définition, et l'étiqueter `sphere` dans le fichier source est au moins aussi défendable que `disk`.
Ce cas apprend surtout que la colonne `shape` contient des choix d'annotation ambigus pour les objets
à la frontière entre deux formes proches, pas que le modèle raisonne mal.

**Relevé hésitant (`changing`, prédictions `light` 26,5 % / `circle` 26,5 %, quasiment à égalité).**
Le témoignage est technique et daté (« hover vibrating white emitting beams and dots of 44 from 03 30
04 30 6 7 2008 over annapolis 44 md ») : les mots qui soutiennent le plus `light` sont `dots`, `white`
et `beams`, cohérents avec un point lumineux. Mais rien dans le vocabulaire de surface ne porte l'idée
de **changement au cours du temps** que porte la forme réelle `changing` — c'est un concept narratif
(l'objet change d'aspect), pas un objet statique nommable par un mot isolé, et un modèle qui ne fait
que moyenner des embeddings de mots n'a struct­urellement aucune prise dessus.

## 11. Export des résultats

In [10]:
def resultat_vers_dataframe(resultat, index_relevé, forme_reelle):
    return pd.DataFrame({
        "index_relevé": index_relevé,
        "mot": resultat["tokens"],
        "score_attribution": resultat["scores"],
        "forme_reelle": forme_reelle,
        "forme_predite": resultat["classe_predite"],
    })

pd.concat([
    resultat_vers_dataframe(resultat_reussi, idx_reussi, forme_reelle_reussi),
    resultat_vers_dataframe(resultat_rate, idx_rate, forme_reelle_rate),
    resultat_vers_dataframe(resultat_hesitant, idx_hesitant, forme_reelle_hesitant),
]).to_csv(PHASE9_DIR / "attributions_mot_par_mot.csv", index=False)

with open(PHASE9_DIR / "relevé_réussi.html", "w", encoding="utf-8") as f:
    f.write(html_reussi)
with open(PHASE9_DIR / "relevé_raté.html", "w", encoding="utf-8") as f:
    f.write(html_rate)
with open(PHASE9_DIR / "relevé_hésitant.html", "w", encoding="utf-8") as f:
    f.write(html_hesitant)

resume_phase9 = pd.DataFrame([{
    "index_reussi": idx_reussi, "forme_reelle_reussi": forme_reelle_reussi,
    "forme_predite_reussi": resultat_reussi["classe_predite"], "proba_reussi": resultat_reussi["proba_predite"],
    "index_rate": idx_rate, "forme_reelle_rate": forme_reelle_rate,
    "forme_predite_rate": resultat_rate["classe_predite"], "proba_rate": resultat_rate["proba_predite"],
    "index_hesitant": idx_hesitant, "forme_reelle_hesitant": forme_reelle_hesitant,
    "top2_hesitant": str(top2_hesitant),
}])
resume_phase9.to_csv(PHASE9_DIR / "resume_phase9.csv", index=False)
resume_phase9


,index_reussi,forme_reelle_reussi,forme_predite_reussi,proba_reussi,index_rate,forme_reelle_rate,forme_predite_rate,proba_rate,index_hesitant,forme_reelle_hesitant,top2_hesitant
0,631,disk,disk,1.0,33570,sphere,disk,0.999991,60063,changing,"[('light', 0.2647642493247986), ('circle', 0.2..."
